In [0]:
%python
from pyspark.sql.functions import col, to_date

# Bronze se read
stores = spark.table("workspace.rp_bronze.stores")
products = spark.table("workspace.rp_bronze.products")
sales = spark.table("workspace.rp_bronze.sales_daily")

# Silver - Cleaning
stores_s = stores.dropDuplicates(["store_id"]).filter(col("store_id").isNotNull())
products_s = products.dropDuplicates(["product_id"]).filter(col("product_id").isNotNull())
sales_s = sales.filter(col("qty") > 0).filter(col("amount").isNotNull())
sales_s = sales_s.withColumn("sale_date", to_date(col("sale_date")))

# Silver me save
stores_s.write.mode("overwrite").saveAsTable("workspace.rp_silver.stores")
products_s.write.mode("overwrite").saveAsTable("workspace.rp_silver.products")
sales_s.write.mode("overwrite").saveAsTable("workspace.rp_silver.sales_daily")

print("M3 DONE - Silver tables created")
print(f"Sales count: {sales_s.count()}")